# CommercePulse — Análise de Vendedores com Concentração de Atrasos

Este notebook investiga quais vendedores concentram a maior parte dos atrasos de entrega,
aplicando o **Princípio de Pareto (80/20)** para identificar oportunidades de melhoria
logística com ações direcionadas.

---

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import sys

# Configuração de caminhos
BASE_DIR = Path().resolve().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
sys.path.insert(0, str(BASE_DIR))
from dashboard.utils import compute_seller_metrics, get_delivered_orders, get_delivered_order_sellers

# Carregar base processada
df = pd.read_csv(PROCESSED_DIR / "commercepulse_orders_items.csv")

date_cols = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"Base carregada: {df.shape[0]:,} itens, {df['order_id'].nunique():,} pedidos")

2026-07-15 03:57:57.465 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-07-15 03:57:57.468 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-07-15 03:57:57.470 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Base carregada: 112,650 itens, 98,666 pedidos


## 1. Filtrar Pedidos Entregues

In [2]:
delivered_orders = get_delivered_orders(df)
delivered = get_delivered_order_sellers(df)

print(f"Pedidos entregues: {len(delivered_orders):,}")
print(f"Taxa de atraso global: {delivered_orders['is_late'].mean()*100:.1f}%")
print(f"Total de pedidos atrasados: {int(delivered_orders['is_late'].sum()):,}")

Pedidos entregues: 96,470
Taxa de atraso global: 8.1%
Total de pedidos atrasados: 7,826


## 2. Métricas por Vendedor

In [3]:
seller_metrics = compute_seller_metrics(df)

print(f"Total de vendedores: {len(seller_metrics):,}")
print(f"Vendedores com atrasos: {(seller_metrics['late_orders'] > 0).sum():,}")
seller_metrics.describe()

Total de vendedores: 3,095
Vendedores com atrasos: 1,390


,total_items,total_orders,total_revenue,delivered_orders,late_orders,avg_review,avg_delivery_days,delay_rate,avg_delay
count,3095.000000,3095.000000,3095.000000,2970.000000,2970.000000,2965.000000,2970.000000,2970.000000,2970.000000
mean,36.397415,32.313409,4391.484233,32.932997,2.641077,4.176139,11.749533,0.084198,0.836739
std,119.193461,105.139763,13921.997192,105.406186,9.376367,0.774781,7.161293,0.168742,4.097137
min,1.000000,1.000000,3.500000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000
25%,2.000000,2.000000,208.850000,2.000000,0.000000,3.968153,8.000000,0.000000,0.000000
50%,8.000000,6.000000,821.480000,7.000000,0.000000,4.281250,10.721895,0.000000,0.000000
75%,24.000000,21.500000,3280.830000,22.000000,2.000000,4.666667,13.781401,0.101308,0.552549
max,2033.000000,1854.000000,229472.630000,1819.000000,195.000000,5.000000,189.000000,1.000000,167.000000


## 3. Distribuição da Taxa de Atraso por Vendedor

In [4]:
fig = px.histogram(
    seller_metrics[seller_metrics["total_orders"] >= 10],
    x="delay_rate",
    nbins=50,
    color_discrete_sequence=["#636EFA"],
    labels={"delay_rate": "Taxa de Atraso", "count": "Qtd Vendedores"},
    title="Distribuição da Taxa de Atraso por Vendedor (mín. 10 pedidos)",
    template="plotly_dark",
    height=400,
)
fig.update_layout(xaxis_tickformat=".0%")
fig.show()

## 4. Top 20 Vendedores com Maiores Taxas de Atraso

Filtramos vendedores com volume mínimo de 30 pedidos para evitar ruído estatístico.

In [5]:
min_orders = 30
sellers_vol = seller_metrics[seller_metrics["total_orders"] >= min_orders].copy()
sellers_vol = sellers_vol.sort_values("delay_rate", ascending=False)

top20 = sellers_vol.head(20)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=top20["seller_id"].str[:8] + "...",
    y=top20["delay_rate"] * 100,
    marker=dict(
        color=top20["delay_rate"] * 100,
        colorscale="Reds",
        showscale=True,
        colorbar=dict(title="Taxa de<br>Atraso (%)"),
    ),
    text=top20["delay_rate"].apply(lambda x: f"{x*100:.1f}%"),
    textposition="outside",
))
fig.update_layout(
    title=f"Top 20 Vendedores com Maiores Taxas de Atraso (mín. {min_orders} pedidos)",
    template="plotly_dark", height=500,
    xaxis_title="Vendedor (ID parcial)", yaxis_title="Taxa de Atraso (%)",
    xaxis_tickangle=-45,
)
fig.show()

## 5. Análise de Pareto — Concentração de Atrasos

O princípio de Pareto sugere que **20% das causas geram 80% dos efeitos**.
Vamos verificar se poucos vendedores são responsáveis pela maioria dos atrasos.

In [6]:
pareto = seller_metrics[seller_metrics["late_orders"] > 0].sort_values("late_orders", ascending=False).reset_index(drop=True)
pareto["cumulative_late"] = pareto["late_orders"].cumsum()
pareto["cumulative_pct"] = pareto["cumulative_late"] / pareto["late_orders"].sum() * 100

n_80 = int((pareto["cumulative_pct"] <= 80).sum()) + 1
pct_sellers_80 = n_80 / len(seller_metrics) * 100

print(f"Total de vendedores com atrasos: {len(pareto):,}")
print(f"Vendedores responsáveis por 80% dos atrasos: {n_80} ({pct_sellers_80:.1f}% do total)")

Total de vendedores com atrasos: 1,390
Vendedores responsáveis por 80% dos atrasos: 423 (13.7% do total)


In [7]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=list(range(1, len(pareto) + 1)),
        y=pareto["late_orders"],
        name="Pedidos Atrasados",
        marker_color="#EF553B",
        opacity=0.7,
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=list(range(1, len(pareto) + 1)),
        y=pareto["cumulative_pct"],
        name="% Acumulado",
        line=dict(color="#00CC96", width=2.5),
    ),
    secondary_y=True,
)
fig.add_hline(y=80, line_dash="dash", line_color="yellow", opacity=0.5, secondary_y=True)

fig.update_layout(
    title=f"Pareto de Atrasos: {n_80} vendedores ({pct_sellers_80:.0f}%) concentram 80% dos atrasos",
    template="plotly_dark", height=500,
    xaxis_title="Vendedores (ordenados por atrasos)",
    legend=dict(x=0.65, y=0.3),
)
fig.update_yaxes(title_text="Pedidos Atrasados", secondary_y=False)
fig.update_yaxes(title_text="% Acumulado", secondary_y=True)
fig.show()

## 6. Correlação: Taxa de Atraso vs. Nota Média

Vendedores com altas taxas de atraso tendem a ter notas piores?

In [8]:
fig = px.scatter(
    sellers_vol,
    x="delay_rate",
    y="avg_review",
    size="total_orders",
    color="seller_state",
    hover_data=["seller_id", "total_orders", "total_revenue"],
    labels={
        "delay_rate": "Taxa de Atraso",
        "avg_review": "Nota Média",
        "total_orders": "Total de Pedidos",
        "seller_state": "Estado",
    },
    title="Taxa de Atraso vs. Nota Média por Vendedor",
    template="plotly_dark",
    height=550,
)
fig.update_layout(xaxis_tickformat=".0%")
fig.show()

## 7. Taxa de Atraso por Estado do Vendedor

In [9]:
state_seller = delivered.groupby("seller_state").agg(
    total_sellers=("seller_id", "nunique"),
    avg_delay_rate=("is_late", "mean"),
    total_orders=("order_id", "nunique"),
    avg_review=("review_score", "mean"),
).reset_index().sort_values("avg_delay_rate", ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=state_seller["seller_state"],
    y=state_seller["avg_delay_rate"] * 100,
    marker=dict(
        color=state_seller["avg_delay_rate"] * 100,
        colorscale="YlOrRd",
    ),
    text=state_seller["avg_delay_rate"].apply(lambda x: f"{x*100:.1f}%"),
    textposition="outside",
))
fig.update_layout(
    title="Taxa de Atraso Média por Estado do Vendedor",
    template="plotly_dark", height=500,
    xaxis_title="Estado", yaxis_title="Taxa de Atraso (%)",
)
fig.show()

## 8. Conclusões

### Principais achados:

1. **Pareto confirmado:** Poucos vendedores concentram a maioria dos atrasos. Ações direcionadas a esse grupo teriam impacto desproporcional na melhoria global.

2. **Atraso está associado à satisfação:** Vendedores associados a mais atrasos apresentam notas inferiores; a análise não demonstra causalidade.

3. **Concentração geográfica:** Alguns estados apresentam taxas de atraso sistematicamente maiores, possivelmente por problemas de infraestrutura logística.

### Recomendações:

- **Programa de monitoramento**: Alertas automáticos para vendedores com taxa de atraso > 15%.
- **Plano de ação progressivo**: Aviso → penalização de ranking → suspensão temporária.
- **Incentivo logístico**: Apoiar vendedores de estados com alta taxa de atraso com soluções de fulfillment regionais.